# Model 3 — Simplified EEG CNN + MLP Architecture
**Multimodal Intraoperative Prediction System — Model 3 of 5**

Encodes brain electrical state from raw EEG waveforms (`BIS/EEG1_WAV`, `BIS/EEG2_WAV`) and BIS scalar channels.
- **Backbone**: Lightweight 1D CNN → Global Average Pool → MLP fusion
- **No MSM pretraining** — removed to eliminate the primary NaN source
- **Stage 1**: Direct supervised training for IOH (primary) and N/H (secondary) labels
- **SQI gate**: windows with `BIS/SQI < 70` excluded from loss
- **NaN filling**: interpolate → forward-fill → backward-fill
- **EEG sample rate**: 128 Hz | Window: 30 sec

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Imports and seed setting
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import gc
import logging
import math
import os
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import stft as scipy_stft
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import vitaldb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. CONFIG — simplified architecture, same data/label contracts
# ─────────────────────────────────────────────────────────────────────────────
CONFIG: Dict = {
    # ── Identity ─────────────────────────────────────────────────────────────
    "model_name": "model_3",
    "log_file": "model_3.log",

    # ── VitalDB tracks ────────────────────────────────────────────────────────
    "wav_tracks": ["BIS/EEG1_WAV", "BIS/EEG2_WAV"],
    "wav_interval": 1 / 128,

    # BIS scalar channels (BIS/BIS kept; ART_MBP and HR needed for labels)
    "scalar_tracks": [
        "BIS/BIS", "BIS/SQI",
        "Solar8000/ART_MBP",
        "Solar8000/HR",
    ],
    "scalar_interval": 1,

    "clinical_params": ["anestart", "aneend", "opstart", "opend", "ane_type"],

    # ── Case eligibility ──────────────────────────────────────────────────────
    "min_case_duration_sec": 1800,
    "min_sqi_coverage": 0.70,
    "sqi_train_threshold": 70,

    # ── Window ───────────────────────────────────────────────────────────────
    "history_sec": 1800,       # 30-min = 1800 s at 1 Hz
    "stride_sec": 60,
    "max_nan_frac": 0.20,
    "max_fill_gap_sec": 60,

    # ── IOH label ─────────────────────────────────────────────────────────────
    "ioh_thresholds_mmhg": 65.0,
    "ioh_min_duration_sec": 60,
    "ioh_horizons_sec": [300, 600, 900],

    # ── N/H label ─────────────────────────────────────────────────────────────
    "nh_n_classes": 3,
    "nh_label_smoothing": 0.10,
    "nh_focal_gamma": 2.0,
    "nh_hr_high": 90,
    "nh_bis_low": 60,
    "nh_bis_high": 65,
    "nh_hr_rise_frac": 0.15,

    # ── EEG waveform ─────────────────────────────────────────────────────────
    "eeg_fs": 128,
    "eeg_window_sec": 30,
    "eeg_samples_per_step": 3840,   # 30 s × 128 Hz
    "eeg_clip_uv": 500.0,

    # ── STFT spectral features ────────────────────────────────────────────────
    "stft_win_sec": 4,
    "stft_overlap": 0.5,
    "stft_bands": {
        "delta": (0.5, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta":  (13.0, 30.0),
    },
    "bsr_window_sec": 63,
    "bsr_amplitude_uv": 5.0,

    # ── Simplified CNN encoder ────────────────────────────────────────────────
    # 4 conv layers: fast, stable, no dilation complexity
    "cnn_channels": [16, 32, 64, 64],  # progressively wider
    "cnn_kernel_size": 7,
    "cnn_out_dim": 64,

    # ── MLP fusion ────────────────────────────────────────────────────────────
    "mlp_hidden_dim": 256,
    "mlp_dropout": 0.3,
    "embedding_dim": 128,

    # ── Scalar feature count ─────────────────────────────────────────────────
    # scalar_tracks = 4 channels; spectral = 6 (4 bands + SEF95 + BSR)
    "n_scalar_features": 4,
    "n_spectral_features": 6,

    # ── Training ──────────────────────────────────────────────────────────────
    "sup_epochs": 50,
    "sup_lr": 1e-4,
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "early_stopping_patience": 8,
    "weight_decay": 1e-4,
    "max_grad_norm": 1.0,

    # ── Data split ────────────────────────────────────────────────────────────
    "val_frac": 0.10,
    "test_frac": 0.10,
    "seed": 42,

    # ── Output files ──────────────────────────────────────────────────────────
    "pred_output": "prediction_model_3.png",
    "checkpoint_sup": "ckpt_model_3_sup.pt",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Logging setup
# ─────────────────────────────────────────────────────────────────────────────
def setup_logging(log_file: str) -> logging.Logger:
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    logger = logging.getLogger("model_3")
    logger.setLevel(logging.DEBUG)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    fh = logging.FileHandler(log_file, mode='w')
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

log = setup_logging(CONFIG["log_file"])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4a. Clinical metadata loader
# ─────────────────────────────────────────────────────────────────────────────
def load_clinical_metadata(config: Dict) -> pd.DataFrame:
    log.info("Loading clinical metadata ...")
    caseids = list(range(1, 6389))
    try:
        df = pd.read_csv("clinical_data.csv")
        df = df[df["caseid"].isin(caseids)]
        cols = ["caseid"] + config["clinical_params"]
        df = df[cols]
    except Exception as exc:
        log.error(f"load_clinical_data failed: {exc}")
        raise
    log.info(f"Clinical metadata loaded: {len(df)} rows")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 4b. Waveform data loader — EEG (128 Hz) + scalars (1 Hz)
# ─────────────────────────────────────────────────────────────────────────────
def load_waveform_data(
    caseids: List[int],
    config: Dict,
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray]]:
    wav_data: Dict[int, np.ndarray] = {}
    scalar_data: Dict[int, np.ndarray] = {}
    wav_tracks    = config["wav_tracks"]
    scalar_tracks = config["scalar_tracks"]
    wav_interval    = config["wav_interval"]
    scalar_interval = config["scalar_interval"]

    for caseid in tqdm(caseids, desc="Loading EEG waveforms"):
        try:
            arr_wav = vitaldb.load_case(caseid=caseid, track_names=wav_tracks, interval=wav_interval)
        except Exception as exc:
            log.warning(f"caseid {caseid}: wav load failed — {exc}")
            arr_wav = None
        if arr_wav is None:
            continue
        try:
            arr_scalar = vitaldb.load_case(caseid=caseid, track_names=scalar_tracks, interval=scalar_interval)
        except Exception as exc:
            log.warning(f"caseid {caseid}: scalar load failed — {exc}")
            del arr_wav; gc.collect(); continue
        if arr_scalar is None:
            del arr_wav; gc.collect(); continue
        wav_data[caseid]    = arr_wav.astype(np.float32)
        scalar_data[caseid] = arr_scalar.astype(np.float32)

    log.info(f"Loaded waveform data for {len(wav_data)} cases")
    return wav_data, scalar_data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. NaN filling — interpolate → forward-fill → backward-fill
#    Applied column-wise on 2D arrays (T, F)
# ─────────────────────────────────────────────────────────────────────────────
def fill_nans(arr: np.ndarray, max_gap_sec: int = 60) -> np.ndarray:
    """Fill NaN gaps per column: interpolate short gaps, then ffill/bfill edges.

    Strategy (per column):
    1. Linear interpolation for gaps <= max_gap_sec samples
    2. Forward-fill any remaining leading/trailing NaNs
    3. Backward-fill any still-remaining NaNs (e.g. column starts with NaN)
    """
    out = arr.copy()
    T, F = out.shape
    for f in range(F):
        s = pd.Series(out[:, f])
        # Step 1: interpolate gaps up to max_gap_sec
        s = s.interpolate(method='linear', limit=max_gap_sec, limit_direction='both')
        # Step 2: forward-fill residual edge NaNs
        s = s.ffill()
        # Step 3: backward-fill anything still NaN at the start
        s = s.bfill()
        out[:, f] = s.values
    return out


def clip_physiological(arr: np.ndarray) -> np.ndarray:
    """Clip implausible scalar values to NaN (column order matches scalar_tracks).

    Column order: BIS/BIS, BIS/SQI, Solar8000/ART_MBP, Solar8000/HR
    """
    out = arr.copy()
    bounds = [
        (0, 100),    # BIS
        (0, 100),    # SQI
        (20, 200),   # MAP mmHg
        (20, 250),   # HR bpm
    ]
    for i, (lo, hi) in enumerate(bounds[:out.shape[1]]):
        col = out[:, i]
        out[:, i] = np.where((col < lo) | (col > hi), np.nan, col)
    return out

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Spectral feature computation (128 Hz EEG)
# ─────────────────────────────────────────────────────────────────────────────
def compute_spectral_features(eeg_window: np.ndarray, config: Dict) -> np.ndarray:
    """Returns [delta, theta, alpha, beta, SEF95, BSR] — shape (6,)."""
    fs      = config["eeg_fs"]
    bands   = config["stft_bands"]
    bsr_amp = config["bsr_amplitude_uv"]
    nperseg = min(int(config["stft_win_sec"] * fs), len(eeg_window))
    nperseg = max(nperseg, 4)
    noverlap = int(nperseg * 0.5)
    try:
        freqs, _, Zxx = scipy_stft(
            eeg_window, fs=fs,
            window='hann', nperseg=nperseg, noverlap=noverlap,
        )
        power      = np.abs(Zxx) ** 2
        mean_power = power.mean(axis=1)
        band_powers = []
        for _, (flo, fhi) in bands.items():
            mask = (freqs >= flo) & (freqs < fhi)
            bp   = mean_power[mask].sum() if mask.any() else 0.0
            band_powers.append(float(np.log1p(bp)))
        total = mean_power.sum()
        if total > 0:
            cum = np.cumsum(mean_power)
            idx = np.searchsorted(cum, 0.95 * total)
            sef95 = float(freqs[min(idx, len(freqs) - 1)])
        else:
            sef95 = 0.0
    except Exception:
        band_powers, sef95 = [0.0, 0.0, 0.0, 0.0], 0.0

    bsr_samples = int(config["bsr_window_sec"] * fs)
    seg = eeg_window[-bsr_samples:] if len(eeg_window) >= bsr_samples else eeg_window
    bsr = float((np.abs(seg) < bsr_amp).mean()) if len(seg) > 0 else 0.0
    return np.array(band_powers + [sef95, bsr], dtype=np.float32)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Label construction — IOH and N/H
# ─────────────────────────────────────────────────────────────────────────────
def max_run_length(arr: np.ndarray) -> int:
    if arr.sum() == 0:
        return 0
    max_run = cur = 0
    for v in arr:
        if v:
            cur += 1; max_run = max(max_run, cur)
        else:
            cur = 0
    return max_run


def build_ioh_labels(map_series: np.ndarray, config: Dict) -> Dict[str, np.ndarray]:
    threshold = config["ioh_thresholds_mmhg"]
    min_dur   = config["ioh_min_duration_sec"]
    labels: Dict[str, np.ndarray] = {}
    for horizon_sec in config["ioh_horizons_sec"]:
        n   = len(map_series)
        lbl = np.zeros(n, dtype=np.float32)
        for t in range(n - horizon_sec):
            window = map_series[t: t + horizon_sec]
            below  = (window < threshold).astype(int)
            if max_run_length(below) >= min_dur:
                lbl[t] = 1.0
        labels[f'ioh_{horizon_sec // 60}'] = lbl
    return labels


def build_nh_labels(
    bis_series: np.ndarray, hr_series: np.ndarray, config: Dict
) -> np.ndarray:
    """3-class N/H: 0=adequate, 1=nociception excess, 2=hypnosis excess."""
    n      = len(bis_series)
    labels = np.zeros(n, dtype=np.int64)
    hr_high      = config["nh_hr_high"]
    bis_low      = config["nh_bis_low"]
    bis_high     = config["nh_bis_high"]
    hr_rise_frac = config["nh_hr_rise_frac"]
    lookback     = 300
    for t in range(lookback, n):
        bis_t, hr_t = bis_series[t], hr_series[t]
        if np.isnan(bis_t) or np.isnan(hr_t):
            continue
        if hr_t > hr_high and bis_t < bis_low:
            labels[t] = 1; continue
        hr_past = hr_series[t - lookback: t]
        if not np.all(np.isnan(hr_past)):
            hr_mean = float(np.nanmean(hr_past))
            if hr_mean > 0 and bis_t > bis_high and (hr_t - hr_mean) / hr_mean > hr_rise_frac:
                labels[t] = 2
    return labels

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. Window generation (30-sec EEG window, 30-min scalar history)
# ─────────────────────────────────────────────────────────────────────────────
def generate_windows(
    wav_data: Dict[int, np.ndarray],
    scalar_data: Dict[int, np.ndarray],
    clinical_df: pd.DataFrame,
    config: Dict,
) -> List[Dict]:
    windows: List[Dict] = []
    history_sec          = config["history_sec"]
    stride_sec           = config["stride_sec"]
    max_nan_frac         = config["max_nan_frac"]
    eeg_samples_per_step = config["eeg_samples_per_step"]
    eeg_fs               = config["eeg_fs"]

    # Column indices for scalar_tracks: BIS, SQI, MAP, HR
    bis_col, sqi_col, map_col, hr_col = 0, 1, 2, 3

    clinical_df = clinical_df.set_index('caseid')

    for caseid, arr_scalar in tqdm(scalar_data.items(), desc='Generating windows'):
        arr_wav = wav_data.get(caseid)
        if arr_wav is None or caseid not in clinical_df.index:
            continue

        T_scalar = arr_scalar.shape[0]
        if T_scalar < config["min_case_duration_sec"]:
            del arr_wav; gc.collect(); continue

        # SQI coverage check
        sqi = arr_scalar[:, sqi_col]
        valid = ~np.isnan(sqi)
        sqi_coverage = (sqi[valid] >= config['sqi_train_threshold']).mean() if valid.sum() > 0 else 0.0
        if sqi_coverage < config["min_sqi_coverage"]:
            del arr_wav; gc.collect(); continue

        # Pre-process scalar: clip → NaN fill
        arr_scalar = clip_physiological(arr_scalar)
        arr_scalar = fill_nans(arr_scalar, max_gap_sec=config["max_fill_gap_sec"])

        # Labels
        ioh_labels = build_ioh_labels(arr_scalar[:, map_col], config)
        nh_labels  = build_nh_labels(arr_scalar[:, bis_col], arr_scalar[:, hr_col], config)

        # Spectral features per 1-Hz step (EEG channel 0)
        eeg_ch0 = arr_wav[:, 0]
        spectral_features = np.zeros((T_scalar, 6), dtype=np.float32)
        for t in range(T_scalar):
            wav_end = int((t + 1) * eeg_fs)
            wav_start = max(0, wav_end - eeg_samples_per_step)
            if wav_end > len(eeg_ch0):
                break
            spectral_features[t] = compute_spectral_features(eeg_ch0[wav_start:wav_end], config)

        # Sliding windows
        for t_end in range(history_sec, T_scalar, stride_sec):
            t_start    = t_end - history_sec
            win_scalar = arr_scalar[t_start:t_end, :]

            # Discard if too many NaNs in any channel
            if any(np.isnan(win_scalar[:, f]).mean() > max_nan_frac for f in range(win_scalar.shape[1])):
                continue
            win_scalar = np.nan_to_num(win_scalar, nan=0.0)

            # Raw EEG (3840, 2) for last 30 s
            wav_end_s   = int(t_end * eeg_fs)
            wav_start_s = max(0, wav_end_s - eeg_samples_per_step)
            if wav_end_s > arr_wav.shape[0]:
                continue
            win_wav = arr_wav[wav_start_s:wav_end_s, :]
            if win_wav.shape[0] < eeg_samples_per_step:
                continue
            win_wav = np.nan_to_num(win_wav, nan=0.0)
            win_wav = np.clip(win_wav, -config["eeg_clip_uv"], config["eeg_clip_uv"])

            win_spectral = spectral_features[t_start:t_end, :]

            # SQI mask (1.0 where SQI >= threshold)
            sqi_win  = arr_scalar[t_start:t_end, sqi_col]
            sqi_mask = (~np.isnan(sqi_win) & (sqi_win >= config['sqi_train_threshold'])).astype(np.float32)
            sqi_end  = float(arr_scalar[t_end - 1, sqi_col]) if not np.isnan(arr_scalar[t_end - 1, sqi_col]) else 0.0

            windows.append({
                'caseid':       caseid,
                'window_start': t_start,
                'win_scalar':   win_scalar.astype(np.float32),     # (1800, 4)
                'win_wav':      win_wav.astype(np.float32),        # (3840, 2)
                'win_spectral': win_spectral.astype(np.float32),   # (1800, 6)
                'sqi_mask':     sqi_mask,                          # (1800,)
                'sqi_end':      sqi_end,
                'label_ioh_5':  float(ioh_labels['ioh_5'][t_end])  if t_end < len(ioh_labels['ioh_5'])  else 0.0,
                'label_ioh_10': float(ioh_labels['ioh_10'][t_end]) if t_end < len(ioh_labels['ioh_10']) else 0.0,
                'label_ioh_15': float(ioh_labels['ioh_15'][t_end]) if t_end < len(ioh_labels['ioh_15']) else 0.0,
                'label_nh':     int(nh_labels[t_end]) if t_end < len(nh_labels) else 0,
            })

        del arr_wav; del wav_data[caseid]; gc.collect()

    log.info(f"Total windows: {len(windows)}")
    return windows

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. Scalers + Dataset
# ─────────────────────────────────────────────────────────────────────────────
def fit_and_apply_scalers(
    train_windows: List[Dict],
    val_windows:   List[Dict],
    test_windows:  List[Dict],
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """Fit StandardScaler on training data; apply to all splits."""
    def _stack(ws, key):
        arrs = [w[key] for w in ws]
        return np.concatenate([a.reshape(-1, a.shape[-1]) for a in arrs], axis=0)

    sc_scalar   = StandardScaler().fit(_stack(train_windows, 'win_scalar'))
    sc_spectral = StandardScaler().fit(_stack(train_windows, 'win_spectral'))
    all_wav     = np.concatenate([w['win_wav'].reshape(-1, 2) for w in train_windows], axis=0)
    eeg_mean    = all_wav.mean(axis=0)
    eeg_std     = all_wav.std(axis=0).clip(min=1e-6)

    def _apply(ws):
        for w in ws:
            T, F   = w['win_scalar'].shape
            T2, F2 = w['win_spectral'].shape
            w['win_scalar']   = sc_scalar.transform(w['win_scalar'].reshape(-1, F)).reshape(T, F).astype(np.float32)
            w['win_spectral'] = sc_spectral.transform(w['win_spectral'].reshape(-1, F2)).reshape(T2, F2).astype(np.float32)
            w['win_wav']      = ((w['win_wav'] - eeg_mean) / eeg_std).astype(np.float32)
        return ws

    log.info('Scalers fitted on training data.')
    return _apply(train_windows), _apply(val_windows), _apply(test_windows)


class EEGDataset(Dataset):
    def __init__(self, windows: List[Dict]) -> None:
        self.windows = windows

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> Dict:
        w = self.windows[idx]
        # Merge scalar (4) + spectral (6) = (1800, 10)
        x = np.concatenate([w['win_scalar'], w['win_spectral']], axis=-1)
        return {
            'x':           torch.FloatTensor(x),
            'wav':         torch.FloatTensor(w['win_wav']),
            'sqi_flag':    torch.FloatTensor(w['sqi_mask']),
            'sqi_end':     torch.FloatTensor([w['sqi_end']]),
            'label_ioh_5':  torch.FloatTensor([w['label_ioh_5']]),
            'label_ioh_10': torch.FloatTensor([w['label_ioh_10']]),
            'label_ioh_15': torch.FloatTensor([w['label_ioh_15']]),
            'label_nh':     torch.LongTensor([w['label_nh']]),
            'caseid':       w['caseid'],
            'window_start': w['window_start'],
        }


def build_dataloaders(
    train_ds: EEGDataset, val_ds: EEGDataset, test_ds: EEGDataset, config: Dict
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    bs = config["batch_size"]
    kw = dict(num_workers=0, pin_memory=True)
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  drop_last=True, **kw)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, **kw)
    test_dl  = DataLoader(test_ds,  batch_size=bs, shuffle=False, **kw)
    log.info(f'DataLoaders: train={len(train_dl)} val={len(val_dl)} test={len(test_dl)}')
    return train_dl, val_dl, test_dl

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. Simplified Model Architecture
#
# Design:
#   EEG branch  : 4-layer 1D CNN → Global Avg Pool → 64-dim vector
#   Scalar branch: mean-pool over time → 10-dim vector → Linear → 64-dim
#   Fusion      : concat (128-dim) → MLP → 128-dim embedding
#   Heads       : 3× IOH binary, 1× N/H 3-class
#
# Why simpler is better here:
#   - No dilated WaveNet → no exploding receptive field gradients
#   - No Transformer → no fp16 QK^T overflow that caused NaN chain
#   - No MSM pretraining → one training stage, one optimizer, one scaler
#   - Larger batch possible (8 vs 4) → more stable gradient estimates
# ─────────────────────────────────────────────────────────────────────────────

class EEGCNNBranch(nn.Module):
    """4-layer 1D CNN that encodes a (B, 2, 3840) EEG window to (B, cnn_out_dim)."""

    def __init__(self, in_channels: int = 2, channels: List[int] = None, kernel_size: int = 7) -> None:
        super().__init__()
        if channels is None:
            channels = [16, 32, 64, 64]
        layers = []
        c_in = in_channels
        for c_out in channels:
            layers += [
                nn.Conv1d(c_in, c_out, kernel_size=kernel_size, padding=kernel_size // 2),
                nn.BatchNorm1d(c_out),
                nn.GELU(),
                nn.MaxPool1d(2),   # halve the sequence each layer
            ]
            c_in = c_out
        self.cnn  = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pool → (B, C, 1)
        self.out_dim = channels[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 2, 3840)
        h = self.cnn(x)           # (B, cnn_out_dim, T/16)
        h = self.pool(h)          # (B, cnn_out_dim, 1)
        return h.squeeze(-1)      # (B, cnn_out_dim)


class ScalarBranch(nn.Module):
    """Mean-pool scalar+spectral time-series → linear projection."""

    def __init__(self, in_dim: int = 10, out_dim: int = 64, dropout: float = 0.3) -> None:
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, in_dim)  → mean over T
        return self.proj(x.mean(dim=1))  # (B, out_dim)


class EEGModel(nn.Module):
    """Simplified EEG Model 3.

    Inputs
    ------
    wav       : (B, 2, 3840)  — z-scored raw EEG at 128 Hz
    x_scalar  : (B, T, 10)   — scalar + spectral features over 1800-step history

    Outputs (Dict)
    -------
    embedding : (B, 128)
    ioh_5/10/15 : (B, 1)  — logits
    nh          : (B, 3)  — logits
    """

    def __init__(self, config: Dict) -> None:
        super().__init__()
        cnn_out   = config["cnn_out_dim"]
        mlp_h     = config["mlp_hidden_dim"]
        drop      = config["mlp_dropout"]
        emb_dim   = config["embedding_dim"]
        n_scalar  = config["n_scalar_features"] + config["n_spectral_features"]  # 10

        self.eeg_branch    = EEGCNNBranch(
            in_channels=2,
            channels=config["cnn_channels"],
            kernel_size=config["cnn_kernel_size"],
        )
        self.scalar_branch = ScalarBranch(in_dim=n_scalar, out_dim=cnn_out, dropout=drop)

        self.fusion = nn.Sequential(
            nn.Linear(cnn_out * 2, mlp_h),
            nn.LayerNorm(mlp_h),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_h, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU(),
        )

        self.ioh_head_5  = nn.Linear(emb_dim, 1)
        self.ioh_head_10 = nn.Linear(emb_dim, 1)
        self.ioh_head_15 = nn.Linear(emb_dim, 1)
        self.nh_head     = nn.Linear(emb_dim, config["nh_n_classes"])

        # Initialise heads with small weights to prevent early NaN
        for head in [self.ioh_head_5, self.ioh_head_10, self.ioh_head_15, self.nh_head]:
            nn.init.xavier_uniform_(head.weight, gain=0.1)
            nn.init.zeros_(head.bias)

    def encode(self, wav: torch.Tensor, x_scalar: torch.Tensor) -> torch.Tensor:
        eeg_feat    = self.eeg_branch(wav.permute(0, 2, 1))   # (B, 2, 3840) → permute → (B, 2, 3840)
        # wav from DataLoader is (B, 3840, 2); permute to (B, 2, 3840) for Conv1d
        scalar_feat = self.scalar_branch(x_scalar)            # (B, 64)
        fused       = torch.cat([eeg_feat, scalar_feat], dim=-1)  # (B, 128)
        return self.fusion(fused)                             # (B, emb_dim)

    def forward(
        self,
        wav:      torch.Tensor,
        x_scalar: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        emb = self.encode(wav, x_scalar)
        return {
            'embedding': emb,
            'ioh_5':     self.ioh_head_5(emb),
            'ioh_10':    self.ioh_head_10(emb),
            'ioh_15':    self.ioh_head_15(emb),
            'nh':        self.nh_head(emb),
        }

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. Loss functions (all fp32, NaN-safe)
# ─────────────────────────────────────────────────────────────────────────────
def ioh_bce_loss(
    pred: torch.Tensor, target: torch.Tensor, sqi_gate: Optional[torch.Tensor] = None
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(pred.float(), target.float(), reduction='none')
    if sqi_gate is not None:
        gate    = sqi_gate.float().view(-1, 1)
        n_valid = gate.sum().clamp(min=1.0)
        return (bce * gate).sum() / n_valid
    return bce.mean()


def focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    gamma: float = 2.0,
    class_weights: Optional[torch.Tensor] = None,
    label_smoothing: float = 0.10,
) -> torch.Tensor:
    n_classes  = logits.size(-1)
    logits_f   = logits.float()
    with torch.no_grad():
        smoothed = torch.zeros_like(logits_f).scatter_(1, targets.unsqueeze(1), 1.0)
        smoothed = smoothed * (1 - label_smoothing) + label_smoothing / n_classes
    log_probs = F.log_softmax(logits_f, dim=-1)
    ce        = -(smoothed * log_probs).sum(dim=-1)
    probs     = torch.exp(log_probs)
    pt        = (probs * F.one_hot(targets, n_classes).float()).sum(dim=-1)
    loss      = (1.0 - pt).pow(gamma) * ce
    if class_weights is not None:
        loss = loss * class_weights.float().clamp(max=10.0)[targets]
    return loss.clamp(max=100.0).mean()


def compute_class_weights(labels_nh: List[int], n_classes: int = 3) -> torch.Tensor:
    arr     = np.array(labels_nh, dtype=np.int64)
    counts  = np.maximum(np.bincount(arr, minlength=n_classes).astype(np.float32), 1.0)
    weights = 1.0 / counts
    return torch.FloatTensor(weights / weights.sum() * n_classes)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. Training and validation loops
# ─────────────────────────────────────────────────────────────────────────────

def _forward_batch(model, batch, device, config):
    wav      = batch['wav'].to(device).permute(0, 2, 1)   # (B, 2, 3840)
    x        = batch['x'].to(device)                      # (B, 1800, 10)
    return model(wav, x), batch


def train_one_epoch(
    model: EEGModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    class_weights_nh: torch.Tensor,
    config: Dict,
    device: torch.device,
) -> float:
    model.train()
    total_loss     = 0.0
    n_batches      = 0
    grad_acc       = config["gradient_accumulation_steps"]
    cw_nh          = class_weights_nh.to(device)
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        out, b = _forward_batch(model, batch, device, config)

        sqi_gate   = b['sqi_end'].to(device)                       # (B, 1)
        label_ioh5 = b['label_ioh_5'].to(device)
        label_ioh10 = b['label_ioh_10'].to(device)
        label_ioh15 = b['label_ioh_15'].to(device)
        label_nh    = b['label_nh'].squeeze(-1).to(device)

        l5  = ioh_bce_loss(out['ioh_5'],  label_ioh5,  sqi_gate)
        l10 = ioh_bce_loss(out['ioh_10'], label_ioh10, sqi_gate)
        l15 = ioh_bce_loss(out['ioh_15'], label_ioh15, sqi_gate)
        lnh = focal_loss(out['nh'], label_nh,
                         gamma=config["nh_focal_gamma"],
                         class_weights=cw_nh,
                         label_smoothing=config["nh_label_smoothing"])
        loss = ((l5 + l10 + l15) / 3.0 + lnh) / grad_acc

        loss_val = loss.item() * grad_acc
        if not math.isfinite(loss_val):
            log.warning(f'Step {step}: non-finite loss {loss_val:.4g} — skipping')
            optimizer.zero_grad()
            continue

        loss.backward()
        if (step + 1) % grad_acc == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss_val
        n_batches  += 1

    # Flush leftover gradients
    if (n_batches % grad_acc) != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def validate(
    model: EEGModel,
    loader: DataLoader,
    class_weights_nh: torch.Tensor,
    config: Dict,
    device: torch.device,
):
    from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score
    model.eval()
    total_loss = 0.0; n_batches = 0
    cw_nh = class_weights_nh.to(device)
    preds  = {k: [] for k in ['ioh_5', 'ioh_10', 'ioh_15']}
    trues  = {k: [] for k in ['ioh_5', 'ioh_10', 'ioh_15']}
    nh_p, nh_t = [], []

    for batch in tqdm(loader, desc='Validate', leave=False):
        out, b = _forward_batch(model, batch, device, config)
        sqi_gate    = b['sqi_end'].to(device)
        label_ioh5  = b['label_ioh_5'].to(device)
        label_ioh10 = b['label_ioh_10'].to(device)
        label_ioh15 = b['label_ioh_15'].to(device)
        label_nh    = b['label_nh'].squeeze(-1).to(device)

        l5  = ioh_bce_loss(out['ioh_5'],  label_ioh5,  sqi_gate)
        l10 = ioh_bce_loss(out['ioh_10'], label_ioh10, sqi_gate)
        l15 = ioh_bce_loss(out['ioh_15'], label_ioh15, sqi_gate)
        lnh = focal_loss(out['nh'], label_nh, gamma=config['nh_focal_gamma'],
                         class_weights=cw_nh, label_smoothing=config['nh_label_smoothing'])
        total_loss += ((l5 + l10 + l15) / 3.0 + lnh).item()
        n_batches  += 1

        for key, logit, lbl in [
            ('ioh_5', out['ioh_5'], label_ioh5),
            ('ioh_10', out['ioh_10'], label_ioh10),
            ('ioh_15', out['ioh_15'], label_ioh15),
        ]:
            preds[key].append(torch.nan_to_num(torch.sigmoid(logit), nan=0.5).cpu().numpy())
            trues[key].append(lbl.cpu().numpy())
        nh_p.append(torch.nan_to_num(out['nh'], nan=0.0).argmax(dim=-1).cpu().numpy())
        nh_t.append(label_nh.cpu().numpy())

    val_loss = total_loss / max(n_batches, 1)
    metrics: Dict[str, float] = {}
    for key in ['ioh_5', 'ioh_10', 'ioh_15']:
        p = np.concatenate(preds[key]).flatten()
        t = np.concatenate(trues[key]).flatten()
        if len(np.unique(t)) < 2:
            metrics[f'auroc_{key}'] = float('nan'); metrics[f'auprc_{key}'] = float('nan')
        else:
            try:
                metrics[f'auroc_{key}'] = float(roc_auc_score(t, p))
                metrics[f'auprc_{key}'] = float(average_precision_score(t, p))
            except ValueError:
                metrics[f'auroc_{key}'] = float('nan'); metrics[f'auprc_{key}'] = float('nan')
    nh_pred = np.concatenate(nh_p); nh_true = np.concatenate(nh_t)
    try:
        metrics['nh_balanced_acc'] = float(balanced_accuracy_score(nh_true, nh_pred))
        metrics['nh_macro_f1']     = float(f1_score(nh_true, nh_pred, average='macro', zero_division=0))
    except ValueError:
        metrics['nh_balanced_acc'] = float('nan'); metrics['nh_macro_f1'] = float('nan')
    return val_loss, metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. EarlyStopping + train_model orchestrator
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 8, checkpoint_path: str = 'best_model.pt') -> None:
        self.patience        = patience
        self.checkpoint_path = checkpoint_path
        self.best_loss       = float('inf')
        self.counter         = 0
        self.should_stop     = False

    def step(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss:
            self.best_loss = val_loss; self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            log.info(f'  ✓ New best val_loss={val_loss:.6f} — checkpoint saved')
        else:
            self.counter += 1
            log.info(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.should_stop = True
                log.info('  Early stopping triggered.')


def train_model(
    model: EEGModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    class_weights_nh: torch.Tensor,
    config: Dict,
    device: torch.device,
) -> EEGModel:
    model = model.to(device)
    log.info("=== Supervised Training (single stage, no MSM) ===")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["sup_lr"], weight_decay=config["weight_decay"]
    )
    # Cosine LR with 2-epoch linear warm-up
    warmup = 2
    def lr_lambda(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        t = (ep - warmup) / max(config['sup_epochs'] - warmup, 1)
        return max(1e-6 / config['sup_lr'], 0.5 * (1 + math.cos(math.pi * t)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    early_stop = EarlyStopping(
        patience=config['early_stopping_patience'],
        checkpoint_path=config['checkpoint_sup'],
    )

    for epoch in range(1, config['sup_epochs'] + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, class_weights_nh, config, device)
        vl_loss, metrics = validate(model, val_loader, class_weights_nh, config, device)
        scheduler.step()
        log.info(
            f"[Ep {epoch}/{config['sup_epochs']}]  "
            f"train={tr_loss:.4f}  val={vl_loss:.4f}  "
            f"auroc_ioh5={metrics.get('auroc_ioh_5', float('nan')):.3f}  "
            f"nh_f1={metrics.get('nh_macro_f1', float('nan')):.3f}  "
            f"({time.time()-t0:.1f}s)"
        )
        early_stop.step(vl_loss, model)
        if early_stop.should_stop:
            break

    model.load_state_dict(torch.load(config['checkpoint_sup'], map_location=device, weights_only=True))
    log.info('Best model loaded from checkpoint.')
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 14. Test evaluation and prediction plot
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_test(
    model: EEGModel,
    test_loader: DataLoader,
    class_weights_nh: torch.Tensor,
    config: Dict,
    device: torch.device,
) -> Dict[str, float]:
    test_loss, metrics = validate(model, test_loader, class_weights_nh, config, device)
    log.info('=== Test Set Evaluation ===')
    log.info(f'  Test loss: {test_loss:.6f}')
    for k, v in metrics.items():
        log.info(f'  {k}: {v:.4f}')
    return metrics


@torch.no_grad()
def plot_predictions(
    model: EEGModel,
    test_loader: DataLoader,
    config: Dict,
    device: torch.device,
    n_cases: int = 5,
) -> None:
    model.eval()
    all_pred, all_true, all_cids = [], [], []

    for batch in test_loader:
        out, b = _forward_batch(model, batch, device, config)
        all_pred.extend(torch.nan_to_num(torch.sigmoid(out['ioh_5']), nan=0.5).cpu().numpy().flatten())
        all_true.extend(b['label_ioh_5'].numpy().flatten())
        all_cids.extend(b['caseid'].tolist() if isinstance(b['caseid'], torch.Tensor) else list(b['caseid']))
        if len(all_pred) >= n_cases * 50:
            break

    unique_cases = list(dict.fromkeys(all_cids))[:n_cases]
    fig, axes   = plt.subplots(n_cases, 1, figsize=(12, 3 * n_cases), squeeze=False)
    for ax_row, cid in zip(axes, unique_cases):
        ax = ax_row[0]
        idxs   = [i for i, c in enumerate(all_cids) if c == cid]
        pred_c = [all_pred[i] for i in idxs]
        true_c = [all_true[i] for i in idxs]
        t      = list(range(len(pred_c)))
        ax.plot(t, pred_c, label='P(IOH-5min)', color='steelblue')
        ax.fill_between(t, true_c, alpha=0.3, color='red', label='IOH true')
        ax.set_ylim(0, 1); ax.set_ylabel('Prob'); ax.set_title(f'Case {cid}')
        ax.legend(loc='upper right', fontsize=8)
    axes[-1][0].set_xlabel('Window index')
    plt.suptitle('Model 3 — EEG IOH-5 Predictions vs Ground Truth', fontsize=13)
    plt.tight_layout()
    plt.savefig(config['pred_output'], dpi=150); plt.close()
    log.info(f"Prediction plot saved → {config['pred_output']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. Patient-level train/val/test split
# ─────────────────────────────────────────────────────────────────────────────
def split_cases(
    all_caseids: List[int],
    val_frac: float = 0.10,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[int], List[int], List[int]]:
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    val_size        = val_frac / (1.0 - test_frac)
    train, val      = train_test_split(train_val,  test_size=val_size,  random_state=seed)
    log.info(f'Split: train={len(train)}, val={len(val)}, test={len(test)}')
    return train, val, test

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 16. Main pipeline
# ─────────────────────────────────────────────────────────────────────────────
def main() -> None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    log.info(f'Using device: {device}')

    # ── 4a: Clinical metadata ─────────────────────────────────────────────
    clinical_df = load_clinical_metadata(CONFIG)

    # ── Case IDs with BIS monitor ─────────────────────────────────────────
    try:
        bis_cases = set(range(1, 21))  # small set for testing
    except AttributeError:
        bis_cases = set(range(1, 6389))
    valid_caseids = sorted(bis_cases)
    log.info(f'Attempting {len(valid_caseids)} BIS cases')

    # ── 4b: Load raw data ─────────────────────────────────────────────────
    wav_data, scalar_data = load_waveform_data(valid_caseids, CONFIG)

    # ── Patient-level split ───────────────────────────────────────────────
    all_caseids = sorted(set(wav_data.keys()) & set(scalar_data.keys()))
    train_ids, val_ids, test_ids = split_cases(
        all_caseids, val_frac=CONFIG['val_frac'], test_frac=CONFIG['test_frac'], seed=CONFIG['seed']
    )

    def _sub(d, keys):
        return {c: d[c] for c in keys if c in d}

    # ── Window generation per split ───────────────────────────────────────
    log.info('Generating training windows ...')
    train_windows = generate_windows(_sub(wav_data, train_ids), _sub(scalar_data, train_ids), clinical_df, CONFIG)
    gc.collect()
    log.info('Generating validation windows ...')
    val_windows   = generate_windows(_sub(wav_data, val_ids),   _sub(scalar_data, val_ids),   clinical_df, CONFIG)
    gc.collect()
    log.info('Generating test windows ...')
    test_windows  = generate_windows(_sub(wav_data, test_ids),  _sub(scalar_data, test_ids),  clinical_df, CONFIG)
    del wav_data, scalar_data; gc.collect()

    if not train_windows:
        log.error('No training windows generated — aborting')
        return

    # ── Scale ─────────────────────────────────────────────────────────────
    train_windows, val_windows, test_windows = fit_and_apply_scalers(
        train_windows, val_windows, test_windows
    )

    # ── Class weights ─────────────────────────────────────────────────────
    class_weights_nh = compute_class_weights(
        [w['label_nh'] for w in train_windows], n_classes=CONFIG['nh_n_classes']
    )
    log.info(f'N/H class weights: {class_weights_nh.tolist()}')

    # ── DataLoaders ───────────────────────────────────────────────────────
    train_ds = EEGDataset(train_windows)
    val_ds   = EEGDataset(val_windows)
    test_ds  = EEGDataset(test_windows)
    train_dl, val_dl, test_dl = build_dataloaders(train_ds, val_ds, test_ds, CONFIG)

    # ── Model ─────────────────────────────────────────────────────────────
    model    = EEGModel(CONFIG)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f'EEGModel parameters: {n_params:,}')

    # ── Train ─────────────────────────────────────────────────────────────
    model = train_model(model, train_dl, val_dl, class_weights_nh, CONFIG, device)

    # ── Evaluate ──────────────────────────────────────────────────────────
    evaluate_test(model, test_dl, class_weights_nh, CONFIG, device)

    # ── Plot ──────────────────────────────────────────────────────────────
    plot_predictions(model, test_dl, CONFIG, device, n_cases=5)

    log.info('model_3 pipeline complete.')


if __name__ == '__main__':
    main()

In [ ]:
main()